# Feature Engineering

## Objective

Feature engineering transforms raw business data into meaningful analytical features.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Load Datasets

In [2]:
PROJECT_ROOT = Path("..")

PROCESSED = PROJECT_ROOT / "data" / "processed"

In [5]:
def load_csv(filename):
    df = pd.read_csv(PROCESSED / filename)
    return df

In [6]:
customers = load_csv("customers.csv")
orders = load_csv("orders.csv")
order_items = load_csv("order_items.csv")
payments = load_csv("payments.csv")
products = load_csv("products.csv")
reviews = load_csv("reviews.csv")
sellers = load_csv("sellers.csv")
geolocation = load_csv("geolocation.csv")

In [7]:
datetime_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in datetime_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

# Order Features

In [9]:
orders["purchase_year"] = (
    orders["order_purchase_timestamp"]
    .dt.year
)

In [11]:
orders["purchase_month"] = (
    orders["order_purchase_timestamp"]
    .dt.month
)

In [12]:
orders["purchase_month_name"] = (
    orders["order_purchase_timestamp"]
    .dt.month_name()
)

In [13]:
orders["purchase_day"] = (
    orders["order_purchase_timestamp"]
    .dt.day
)

In [14]:
orders["purchase_hour"] = (
    orders["order_purchase_timestamp"]
    .dt.hour
)

In [15]:
orders["purchase_weekday"] = (
    orders["order_purchase_timestamp"]
    .dt.day_name()
)

In [16]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] -
    orders["order_purchase_timestamp"]
).dt.days

In [17]:
orders["estimated_delivery_days"] = (
    orders["order_estimated_delivery_date"] -
    orders["order_purchase_timestamp"]
).dt.days

In [20]:
orders["shipping_days"] = (
    orders["order_delivered_customer_date"] -
    orders["order_delivered_carrier_date"]
).dt.days

In [22]:
orders["delivery_delay_days"] = (
    orders["delivery_days"] -
    orders["estimated_delivery_days"]
)

In [23]:
orders["delivery_status"] = np.where(
    orders["delivery_delay_days"] <= 0,
    "On Time",
    "Late"
)

# Product Features

In [25]:
products["product_volume_cm3"] = (
    products["product_length_cm"] *
    products["product_height_cm"] *
    products["product_width_cm"]
)

In [26]:
def classify_product_size(volume):
    if pd.isna(volume):
        return np.nan
    elif volume < 1000:
        return "Small"
    elif volume < 10000:
        return "Medium"
    else:
        return "Large"


products["product_size"] = (
    products["product_volume_cm3"]
    .apply(classify_product_size)
)

In [27]:
def classify_weight(weight):
    if pd.isna(weight):
        return np.nan
    elif weight < 500:
        return "Light"
    elif weight < 2000:
        return "Medium"
    else:
        return "Heavy"


products["weight_category"] = (
    products["product_weight_g"]
    .apply(classify_weight)
)

# Payment Features

In [29]:
payments["is_installment"] = (
    payments["payment_installments"] > 1
)

In [30]:
def classify_installments(value):
    if value == 1:
        return "Single Payment"
    elif value <= 3:
        return "Short Installments"
    elif value <= 6:
        return "Medium Installments"
    else:
        return "Long Installments"


payments["installment_category"] = (
    payments["payment_installments"]
    .apply(classify_installments)
)

In [31]:
def classify_payment(value):
    if pd.isna(value):
        return np.nan
    elif value < 100:
        return "Low"
    elif value < 500:
        return "Medium"
    else:
        return "High"


payments["payment_value_category"] = (
    payments["payment_value"]
    .apply(classify_payment)
)

In [33]:
payments["payment_value_rounded"] = (
    payments["payment_value"]
    .round(2)
)

# Review Features

In [34]:
def review_sentiment(score):
    if pd.isna(score):
        return np.nan
    elif score >= 4:
        return "Positive"
    elif score == 3:
        return "Neutral"
    else:
        return "Negative"


reviews["review_sentiment"] = (
    reviews["review_score"]
    .apply(review_sentiment)
)

In [35]:
def satisfaction_level(score):
    if pd.isna(score):
        return np.nan
    elif score == 5:
        return "Excellent"
    elif score == 4:
        return "Good"
    elif score == 3:
        return "Average"
    elif score == 2:
        return "Poor"
    else:
        return "Very Poor"


reviews["satisfaction_level"] = (
    reviews["review_score"]
    .apply(satisfaction_level)
)

In [36]:
reviews["would_recommend"] = (
    reviews["review_score"] >= 4
)

# Order items Features

In [42]:
order_items["total_item_cost"] = (
    order_items["price"] +
    order_items["freight_value"]
)